In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import json
import pickle
import ast

In [ ]:
# Load Yijun's 5-fold indices
kfold_pt = 'sfold5_split_train_test_indices.json'
with open(kfold_pt, 'r') as file:
    kfold_indices = json.load(file)
print(type(kfold_indices))

KFOLD INDICES STRUCTURE:
Dict
- first level are keys labeled fold_0 to fold_1
    - in each fold_i value is another dictionary where the keys are 'train' and 'test'
        - the value is a list of indices

In [ ]:
# Load in the geojson with data
pt_data_pt = '../data/point_pixel_satbands_gt_epsg4326_v3_removednodata.geojson'
point_data_gdf = gpd.read_file(pt_data_pt)
print(point_data_gdf.columns)

In [ ]:
# Define random forest run function
def run_rf(all_ids, kfold_indices, X, y, hyperparameters, save_preds=False, save_rf=False):
    indices = np.array(all_ids)
    
    fold_mse = []
    fold_r2 = []
    
    for fold, train_test_dict in kfold_indices.items():
        print('Training Fold: ', fold)

        train_indices = train_test_dict['train']
        test_indices = train_test_dict['test']

        train_rows = np.where(np.isin(indices, train_indices))[0]
        test_rows = np.where(np.isin(indices, test_indices))[0]

        X_train = X[train_rows]
        X_test = X[test_rows]
        y_train = y[train_rows]
        y_test = y[test_rows] 

        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        est = hyperparameters['est']
        verbose = hyperparameters['verbose']
        n_jobs = hyperparameters['n_jobs']
        rf = RandomForestRegressor(n_estimators=est, verbose=verbose, n_jobs=n_jobs)
        rf.fit(X_train, y_train.ravel())
        y_pred = rf.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        fold_mse.append(mse)
        fold_r2.append(r2)
        
        fold_num = int(fold.split('_')[-1])
        print(f"Fold {fold_num} MSE: {mse:.4f}")
        print(f"Fold {fold_num} R2: {r2:.4f}")
        
        if save_preds:
            test_ids = indices[np.isin(indices, test_indices)]
            preds = {
                "id": test_ids,  
                "gt": y_test.flatten(), 
                "pred": y_pred  
            }
            df_predictions = pd.DataFrame(preds)
            preds_outpath = f"test_predictions_{fold}.csv"
            df_predictions.to_csv(preds_outpath, index=False)

            print(f"Test predictions saved to {preds_outpath}")
            
        if save_rf:
            model_file = f"rf_weights_{fold}.pkl"
            with open(model_file, "wb") as f:
                pickle.dump(rf, f)
            print(f"Model saved to {model_file}")
        
    return (
        fold_mse, fold_r2
    )

## Experiment Random Forest 11 (RF-11)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: basically satellite channels, env covars like DEM, climate covars
        - B, G, R, rededge1, rededge2, rededge3, nir, swir1, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Organic thickness, o_thick_cum_best (image channel), regression
- Preprocessing: removal of NoData rows

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
def is_invalid(val):
    if val == 'nan':
        return True
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return True
    except (ValueError, SyntaxError):
        pass
    return False
point_data_gdf = point_data_gdf[~point_data_gdf['aksdb_othick_cum_best'].apply(is_invalid)].copy()
point_data_gdf['aksdb_othick_cum_best'] = point_data_gdf['aksdb_othick_cum_best'].astype(float)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in covars
topo_covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
topo_covar_gdf = gpd.read_file(topo_covar_pt)
topo_covar_gdf = topo_covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]
topo_covar_gdf['id'] = topo_covar_gdf['id'].astype('int64')

climate_covar_pt = '../data/point_pixel_climate_v1.csv'
climate_covar_gdf = pd.read_csv(climate_covar_pt)
climate_covar_gdf = climate_covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [ ]:
merged_df = point_data_gdf.merge(topo_covar_gdf, on='id', how='inner')
merged_df = merged_df.merge(climate_covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 
                    'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 
                     'swi_10_band_1', 'tpi_4_band_1', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_othick_cum_best']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    mse, r2
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(mse))),
    "MSE": mse,
    "R2": r2,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_thickness_cum_best/RF-11_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")